# MOD-06 · NB-01 — Causal Thinking: DAGs, Confounding & Identification
### Health Analytics with Python · Module 06: Causal Inference in Health Research
---
**Learning objectives**
- Distinguish association from causation with clinical examples
- Draw and interpret Directed Acyclic Graphs (DAGs)
- Identify confounders, mediators, colliders using graph structure
- Apply the backdoor criterion to find valid adjustment sets
- Simulate confounded data and quantify bias with vs without adjustment

**Estimated time:** 2.5 hours | **Level:** Intermediate | **Libraries:** `networkx`, `numpy`, `scipy`


## 1. Setup & Shared Dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
import os; os.makedirs("/tmp/mod06", exist_ok=True)
plt.rcParams.update({"figure.dpi":120,"figure.facecolor":"white",
                     "axes.spines.top":False,"axes.spines.right":False})

np.random.seed(42); N = 5000
def sigmoid(x): return 1/(1+np.exp(-x))

# Confounders
age          = np.random.normal(60, 12, N).clip(30, 90)
smoking      = np.random.binomial(1, sigmoid(-1 + 0.02*(age-60)), N)
chd_history  = np.random.binomial(1, sigmoid(-2 + 0.03*(age-60)), N)
ldl_baseline = np.random.normal(140 + 20*smoking, 30, N).clip(60, 280)
ses_score    = np.random.normal(0, 1, N)  # socioeconomic (higher=better)

# Treatment: statin use (confounded by LDL, CHD, age, SES)
statin_logit = (-2.0 + 0.025*(ldl_baseline-140) + 1.0*chd_history
                + 0.02*(age-60) - 0.3*ses_score)
statin = np.random.binomial(1, sigmoid(statin_logit), N)

# True causal effect: statin reduces MI risk (log-odds = -0.8)
TRUE_EFFECT = -0.8
mi_logit = (-3.0 + TRUE_EFFECT*statin + 0.03*(age-60)
            + 0.5*smoking + 0.8*chd_history + 0.005*(ldl_baseline-140))
mi = np.random.binomial(1, sigmoid(mi_logit), N)

df = pd.DataFrame({
    "age":age, "smoking":smoking, "chd_history":chd_history,
    "ldl_baseline":ldl_baseline, "ses_score":ses_score,
    "statin":statin, "mi":mi,
})
COVARIATES = ["age","smoking","chd_history","ldl_baseline","ses_score"]
TREATMENT  = "statin"
OUTCOME    = "mi"
print(f"N={N} | Statin={statin.mean()*100:.1f}% | MI rate={mi.mean()*100:.1f}%")
print(f"True log-odds effect: {TRUE_EFFECT} | True OR: {np.exp(TRUE_EFFECT):.3f}")


## 2. Association vs Causation

**Potential Outcomes Framework** (Rubin, 1974):
- Y(1) = outcome *if treated* | Y(0) = outcome *if untreated*
- Individual causal effect = Y(1) - Y(0) — **never** directly observed (fundamental problem)
- ATE = E[Y(1) - Y(0)] = E[Y(1)] - E[Y(0)]

**Key structural variable types:**

| Type | DAG pattern | Action |
|------|-------------|--------|
| Confounder | C → T and C → Y | Must adjust |
| Mediator | T → M → Y | Do NOT adjust (blocks causal path) |
| Collider | T → C ← Y | Do NOT adjust (opens spurious path) |
| Instrument | Z → T, Z ⊥ Y given T | Use for IV estimation |


In [ ]:
try:
    import networkx as nx
    import matplotlib.patches as mpatches
    NETWORKX_OK = True
except ImportError:
    NETWORKX_OK = False
    print("pip install networkx")

if NETWORKX_OK:
    G = nx.DiGraph()
    pos = {
        "Age":(0,1),"Smoking":(0,0),"SES":(0,-1),
        "LDL":(1,0.5),"CHD":(1,-0.5),
        "Statin":(2,0),"MI":(3,0),
    }
    edges = [
        ("Age","LDL"),("Age","CHD"),("Age","MI"),
        ("Smoking","LDL"),("Smoking","MI"),
        ("SES","Smoking"),("SES","Statin"),
        ("LDL","Statin"),("LDL","MI"),
        ("CHD","Statin"),("CHD","MI"),
        ("Statin","MI"),  # causal effect of interest
    ]
    G.add_edges_from(edges)

    colors = {"Statin":"#4878CF","MI":"#D65F5F"}
    node_colors = [colors.get(n,"#6ACC65") for n in G.nodes()]

    fig, ax = plt.subplots(figsize=(12,5))
    nx.draw_networkx(G, pos=pos, ax=ax, node_color=node_colors,
                     node_size=2200, font_size=9, font_weight="bold",
                     edge_color="gray", arrows=True, arrowsize=18,
                     width=1.5, connectionstyle="arc3,rad=0.08")
    ax.legend(handles=[mpatches.Patch(color="#4878CF",label="Treatment"),
                       mpatches.Patch(color="#D65F5F",label="Outcome"),
                       mpatches.Patch(color="#6ACC65",label="Confounder")],
              fontsize=9)
    ax.set_title("Clinical DAG: Statin -> MI with confounders and mediators", fontsize=12)
    ax.axis("off"); plt.tight_layout()
    plt.savefig("/tmp/mod06/dag_statin_mi.png", bbox_inches="tight"); plt.show()
    print("Backdoor paths (non-causal paths from Statin to MI):")
    parents_statin = list(G.predecessors("Statin"))
    for p in parents_statin:
        print(f"  Statin <- {p} -> MI  (confounder: {p})")


## 3. Naive vs Adjusted Effect Estimation

In [ ]:
from sklearn.linear_model import LogisticRegression

# Naive (ignores confounding)
lr_naive = LogisticRegression(max_iter=500).fit(df[["statin"]], df["mi"])
naive_coef = lr_naive.coef_[0][0]

# Adjusted (backdoor adjustment)
lr_adj = LogisticRegression(max_iter=500).fit(df[COVARIATES + ["statin"]], df["mi"])
adj_coef   = lr_adj.coef_[0][0]  # statin coefficient

print(f"True effect         : {TRUE_EFFECT:.3f}  (OR = {np.exp(TRUE_EFFECT):.3f})")
print(f"Naive estimate      : {naive_coef:.3f}  (OR = {np.exp(naive_coef):.3f})")
print(f"Adjusted estimate   : {adj_coef:.3f}  (OR = {np.exp(adj_coef):.3f})")
print(f"Naive bias          : {naive_coef - TRUE_EFFECT:+.3f} log-odds")
print(f"Adjusted bias       : {adj_coef - TRUE_EFFECT:+.3f} log-odds")
print()
print("Interpretation: Positive confounding (sick patients more likely to get statins)")
print("makes the naive estimate look LESS protective than the true causal effect.")


## 4. Collider Bias Demonstration

In [ ]:
# Among hospitalised patients: smoking & alcohol appear correlated
# In truth they are independent — conditioning on hospitalisation (collider) opens bias
from scipy import stats
np.random.seed(7); N_col = 3000
smoking_c  = np.random.normal(0,1,N_col)
alcohol_c  = np.random.normal(0,1,N_col)   # truly independent of smoking
hosp_score = 0.6*smoking_c + 0.6*alcohol_c + np.random.normal(0,0.5,N_col)
hospitalised = (hosp_score > np.percentile(hosp_score, 75)).astype(int)

r_all, p_all = stats.pearsonr(smoking_c, alcohol_c)
r_hosp, p_hosp = stats.pearsonr(smoking_c[hospitalised==1], alcohol_c[hospitalised==1])

fig, axes = plt.subplots(1,2,figsize=(12,4))
for ax, mask, title, color in [
    (axes[0], np.ones(N_col,dtype=bool), f"Full population r={r_all:.3f}", "#4878CF"),
    (axes[1], hospitalised==1,           f"Hospitalised only r={r_hosp:.3f}", "#D65F5F"),
]:
    ax.scatter(smoking_c[mask], alcohol_c[mask], alpha=0.3, s=10, color=color)
    m,b = np.polyfit(smoking_c[mask],alcohol_c[mask],1)
    xl = np.linspace(smoking_c[mask].min(),smoking_c[mask].max(),100)
    ax.plot(xl,m*xl+b,"k-",lw=2)
    ax.set_xlabel("Smoking (latent)"); ax.set_ylabel("Alcohol (latent)")
    ax.set_title(title, fontsize=10)
plt.suptitle("Collider Bias: Conditioning on hospitalisation\n(a collider) creates spurious smoking-alcohol correlation",fontsize=11)
plt.tight_layout()
plt.savefig("/tmp/mod06/collider_bias.png",bbox_inches="tight"); plt.show()


## 5. Backdoor Criterion & Adjustment Sets

In [ ]:
# Formal backdoor criterion check
def backdoor_ok(treatment_parents, adjustment_set, treatment_descendants):
    # Rule 1: No descendant of treatment in adjustment set
    for v in adjustment_set:
        if v in treatment_descendants:
            return False, f"{v} is a descendant of treatment — blocks causal path!"
    # Rule 2: All confounders (parents of treatment) blocked
    unblocked = [p for p in treatment_parents if p not in adjustment_set]
    if unblocked:
        return False, f"Open backdoor path via: {unblocked}"
    return True, "Adjustment set is valid (all backdoor paths blocked)"

# In our DAG: parents of Statin = Age, LDL, CHD, SES (direct confounders)
# Descendants of Statin = MI (outcome only, no mediators explicitly modelled here)
treatment_parents = ["LDL","CHD","Age","SES"]  # from DAG
treatment_desc    = ["MI"]

tests = {
    "Empty set"                        : [],
    "Age only"                         : ["Age"],
    "LDL, CHD, Age, SES (full)"       : ["LDL","CHD","Age","SES"],
    "LDL, CHD, Age, SES, MI (wrong!)" : ["LDL","CHD","Age","SES","MI"],
    "Smoking, CHD, LDL, Age, SES"     : ["Smoking","CHD","LDL","Age","SES"],
}
print(f"{'Adjustment set':45s} {'Valid?':8s} {'Notes'}")
print("-"*90)
for name, adj in tests.items():
    ok, msg = backdoor_ok(treatment_parents, adj, treatment_desc)
    status = "VALID" if ok else "INVALID"
    print(f"  {name:43s} {status:8s} {msg}")


## 6. Knowledge Check
1. In the statin DAG, LDL appears as both a confounder AND a mediator. How do you handle this?
2. A researcher adjusts for post-diagnosis employment status when studying chemotherapy's effect on survival. What type of bias does this introduce?
3. SES is a confounder of statin use. But SES is not measured in most EHR datasets. What is this problem called, and how might you address it?
4. Draw a DAG for the question: "Does exercise reduce depression?" Include BMI, social support, sleep quality, and medication use.
5. Show algebraically why E[Y|X=1] - E[Y|X=0] ≠ E[Y(1)] - E[Y(0)] when there is unmeasured confounding.


In [ ]:
# Exercise 5 — decompose association vs causal effect numerically
# True ATE
ate_true = TRUE_EFFECT  # -0.8 log-odds by design

# What we observe: E[Y|statin=1] - E[Y|statin=0]  (association, not causation)
observed_diff = df.groupby("statin")["mi"].mean()
log_or_naive  = np.log(observed_diff[1]/(1-observed_diff[1])) - np.log(observed_diff[0]/(1-observed_diff[0]))

# Decompose: association = causal effect + confounding bias
print("Decomposition of association vs causation:")
print(f"  Observed log-OR  = {log_or_naive:.3f}")
print(f"  True causal effect = {ate_true:.3f}")
print(f"  Confounding bias   = {log_or_naive - ate_true:+.3f}")
print()
print("  Observed = Causal + Confounding")
print(f"  {log_or_naive:.3f} = {ate_true:.3f} + {log_or_naive-ate_true:.3f}")
print()
print("The bias is POSITIVE: sick patients (higher MI risk) preferentially receive statins,")
print("making statins appear less effective than they truly are.")


---
**Next -> NB-02: Propensity Score Methods**